In [1]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.linear_model import HuberRegressor, Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    precision_recall_fscore_support, average_precision_score
)

import xgboost as xgb
from xgboost import XGBRegressor, XGBClassifier
from IPython.display import display

In [ ]:
warnings.filterwarnings("ignore")

# CONFIG
DATA_PATH   = "processed/chennai_aqi_era5_hourly.parquet"
OUT_DIR     = "huber_processed_leakfree"
SPLIT_TIME  = pd.Timestamp("2025-01-01 00:00:00")
L           = 24
H_LIST      = [1, 3, 6, 12, 24]
TAU_LIST    = [0, 1, 2, 3, 4, 6]
VALID_FRAC  = 0.20
GRAPH_K     = 4

RUN_TUNED   = True
JSO_POP     = 12
JSO_ITERS   = 15
RANDOM_SEED = 1

GATE_MODE = "fixed"

In [ ]:
# HELPERS
def haversine_km_vec(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2.0 * R * np.arcsin(np.sqrt(a))

def bearing_radians(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return np.arctan2(y, x)

def flatten_window_per_node(X):
    S, Lx, N, F = X.shape
    return X.transpose(0, 2, 1, 3).reshape(S * N, Lx * F)

def compute_tail_metrics(y_true, y_pred, percentiles=(90, 95, 99)):
    rows = []
    for p in percentiles:
        thr = np.percentile(y_true, p)
        idx = y_true >= thr
        if idx.sum() == 0:
            rows.append((p, thr, np.nan, np.nan, 0))
        else:
            rows.append((
                p,
                thr,
                mean_absolute_error(y_true[idx], y_pred[idx]),
                np.sqrt(mean_squared_error(y_true[idx], y_pred[idx])),
                int(idx.sum())
            ))
    return rows

def compute_mfb_nmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mfb = np.mean(2.0 * (y_pred - y_true) / (y_pred + y_true + 1e-8))
    nmse = np.sum((y_pred - y_true) ** 2) / (np.sum(y_pred * y_true) + 1e-8)
    return mfb, nmse

def build_fill_values_from_train_timeline(X_train_timeline):
    fill_values = np.nanmedian(X_train_timeline, axis=0)
    global_fill = np.nanmedian(X_train_timeline.reshape(-1, X_train_timeline.shape[-1]), axis=0)
    global_fill = np.where(np.isnan(global_fill), 0.0, global_fill)
    for n in range(fill_values.shape[0]):
        for f in range(fill_values.shape[1]):
            if np.isnan(fill_values[n, f]):
                fill_values[n, f] = global_fill[f]
    fill_values = np.where(np.isnan(fill_values), 0.0, fill_values).astype(np.float32)
    return fill_values

def impute_windows(X, fill_values):
    X_imp = X.copy().astype(np.float32)
    S, Lx, N, F = X_imp.shape
    for s in range(S):
        for n in range(N):
            for f in range(F):
                series = pd.Series(X_imp[s, :, n, f], dtype="float32")
                series = series.ffill().bfill()
                arr = series.to_numpy(dtype=np.float32)
                if np.isnan(arr).any():
                    arr = np.where(np.isnan(arr), fill_values[n, f], arr)
                X_imp[s, :, n, f] = arr
    return X_imp

def build_nodes_edges(df, station_ids, k=4):
    coords = (
        df[["station_id", "lat", "lon"]]
        .drop_duplicates()
        .assign(station_id=lambda d: d["station_id"].astype(str))
    )
    nodes = coords[coords["station_id"].isin(station_ids)].copy()
    nodes = nodes.set_index("station_id").loc[station_ids].reset_index()
    nodes["node_id"] = np.arange(len(nodes))

    coords_arr = nodes[["lat", "lon"]].to_numpy(dtype=float)
    N = len(nodes)

    D = np.zeros((N, N), dtype=float)
    for i in range(N):
        D[i, :] = haversine_km_vec(coords_arr[i, 0], coords_arr[i, 1], coords_arr[:, 0], coords_arr[:, 1])

    sigma = np.median(D[D > 0])
    edges = []
    kk = min(k, N - 1)
    for i in range(N):
        nn = np.argsort(D[i])[1:kk + 1]
        for j in nn:
            w = np.exp(-(D[i, j] ** 2) / (2.0 * sigma ** 2))
            edges.append((i, j, float(w), float(D[i, j])))

    edges_df = pd.DataFrame(edges, columns=["src", "dst", "w_dist", "dist_km"])
    return nodes, edges_df

def build_static_adj_from_nodes(nodes_df, k=4, sigma_km=None, self_loops=False, row_normalize=True):
    coords = nodes_df[["lat", "lon"]].to_numpy(dtype=float)
    N = coords.shape[0]
    D = np.zeros((N, N), dtype=float)
    for i in range(N):
        D[i, :] = haversine_km_vec(coords[i, 0], coords[i, 1], coords[:, 0], coords[:, 1])

    if sigma_km is None:
        sigma_km = np.median(D[D > 0])

    A = np.zeros((N, N), dtype=np.float32)
    for i in range(N):
        nn = np.argsort(D[i])[1:min(k + 1, N)]
        for j in nn:
            A[i, j] = np.float32(np.exp(-(D[i, j] ** 2) / (2.0 * sigma_km ** 2)))

    if self_loops:
        np.fill_diagonal(A, 1.0)

    if row_normalize:
        rs = A.sum(axis=1, keepdims=True)
        A = np.divide(A, rs, out=np.zeros_like(A), where=rs > 0)

    return A

def wind_bearing_toward(u, v):
    return np.arctan2(u, v)


def build_dynamic_adj(u_t, v_t, ctx, alpha=4.0, eps=0.05, gate_mode=None):
    if gate_mode is None:
        gate_mode = globals().get("GATE_MODE", "fixed")
    src, dst = ctx["src"], ctx["dst"]

    if gate_mode == "legacy":
        theta = np.arctan2(v_t[src], u_t[src])
        bearing = ctx["edge_bearing"]
    elif gate_mode in ("fixed", "shuffled"):
        theta = wind_bearing_toward(u_t[dst], v_t[dst])
        key = "edge_bearing_shuffled" if gate_mode == "shuffled" else "edge_bearing"
        if key not in ctx:
            raise KeyError(f"missing ctx key {key!r}")
        bearing = ctx[key] + np.pi
    else:
        raise ValueError(f"unknown gate_mode {gate_mode!r}")

    align = np.cos(theta - bearing)
    gate = 1.0 / (1.0 + np.exp(-alpha * align))
    gate = eps + (1.0 - eps) * gate
    w_dyn = ctx["w_dist"] * gate

    A = np.zeros((ctx["N"], ctx["N"]), dtype=np.float32)
    A[src, dst] = w_dyn.astype(np.float32)
    rs = A.sum(axis=1, keepdims=True)
    A = np.divide(A, rs, out=np.zeros_like(A), where=rs > 0)
    return A
def make_graph_features_dynamic(X, ctx, tau=0, alpha=4.0, eps=0.05, gate_mode=None):
    S, Lx, N, F = X.shape
    Z = np.zeros((S, N, 2 * F), dtype=np.float32)
    tau = int(tau)
    for s in range(S):
        x_now = X[s, -1]
        x_lag = X[s, -1 - tau] if tau > 0 else x_now
        A = build_dynamic_adj(
            x_now[:, ctx["u_idx"]],
            x_now[:, ctx["v_idx"]],
            ctx,
            alpha=alpha,
            eps=eps,
            gate_mode=gate_mode
        )
        agg = A @ x_lag
        Z[s] = np.concatenate([x_now, agg], axis=-1)
    return Z
def make_graph_features_static(X, A_static, tau=0):
    S, Lx, N, F = X.shape
    Z = np.zeros((S, N, 2 * F), dtype=np.float32)
    tau = int(tau)
    for s in range(S):
        x_now = X[s, -1]
        x_lag = X[s, -1 - tau] if tau > 0 else x_now
        agg = A_static @ x_lag
        Z[s] = np.concatenate([x_now, agg], axis=-1)
    return Z

def summarize_regression(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred))
    }

def train_val_split_time_order(X_train_imp, Y_train, M_train, target_times_train, frac=0.2):
    n = X_train_imp.shape[0]
    val_size = max(1, int(np.ceil(frac * n)))
    idx_train = np.arange(0, n - val_size)
    idx_val   = np.arange(n - val_size, n)
    train_pack = {
        "X": X_train_imp[idx_train],
        "Y": Y_train[idx_train],
        "M": M_train[idx_train],
        "times": target_times_train[idx_train]
    }
    val_pack = {
        "X": X_train_imp[idx_val],
        "Y": Y_train[idx_val],
        "M": M_train[idx_val],
        "times": target_times_train[idx_val]
    }
    return train_pack, val_pack

def fit_flat_regressor(model, X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test):
    Xtr = flatten_window_per_node(X_train_imp)
    Xte = flatten_window_per_node(X_test_imp)
    ytr = Y_train.reshape(-1)
    yte = Y_test.reshape(-1)
    mtr = (M_train.reshape(-1) > 0.5)
    mte = (M_test.reshape(-1) > 0.5)

    model.fit(Xtr[mtr], ytr[mtr])
    pred_all = model.predict(Xte)
    return {
        "y_true": yte[mte],
        "y_pred": pred_all[mte],
        "model": model
    }

def fit_static_graph_regressor(model, X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test, A_static, tau):
    Ztr = make_graph_features_static(X_train_imp, A_static, tau=tau)
    Zte = make_graph_features_static(X_test_imp, A_static, tau=tau)

    Xtr = Ztr.reshape(-1, Ztr.shape[-1])
    Xte = Zte.reshape(-1, Zte.shape[-1])
    ytr = Y_train.reshape(-1)
    yte = Y_test.reshape(-1)
    mtr = (M_train.reshape(-1) > 0.5)
    mte = (M_test.reshape(-1) > 0.5)

    model.fit(Xtr[mtr], ytr[mtr])
    pred_all = model.predict(Xte)
    return {
        "y_true": yte[mte],
        "y_pred": pred_all[mte],
        "model": model
    }

def fit_huber_graph(X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test, ctx, tau=0, alpha=4.0, eps=0.05, huber_alpha=1e-4, gate_mode=None):
    Ztr = make_graph_features_dynamic(X_train_imp, ctx, tau=tau, alpha=alpha, eps=eps, gate_mode=gate_mode)
    Zte = make_graph_features_dynamic(X_test_imp, ctx, tau=tau, alpha=alpha, eps=eps, gate_mode=gate_mode)

    Xtr = Ztr.reshape(-1, Ztr.shape[-1])
    Xte = Zte.reshape(-1, Zte.shape[-1])
    ytr = Y_train.reshape(-1)
    yte = Y_test.reshape(-1)
    mtr = (M_train.reshape(-1) > 0.5)
    mte = (M_test.reshape(-1) > 0.5)

    huber = Pipeline([
        ("scaler", StandardScaler()),
        ("huber", HuberRegressor(epsilon=1.35, alpha=huber_alpha, max_iter=500))
    ])
    huber.fit(Xtr[mtr], ytr[mtr])

    pred_train_all = huber.predict(Xtr)
    pred_test_all  = huber.predict(Xte)

    return {
        "y_true": yte[mte],
        "y_pred": pred_test_all[mte],
        "pred_train_all": pred_train_all,
        "pred_test_all": pred_test_all,
        "y_train_all": ytr,
        "y_test_all": yte,
        "mask_train": mtr,
        "mask_test": mte,
        "model": huber
    }

def fit_huber_hybrid(
    X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test,
    ctx, tau=0, alpha=4.0, eps=0.05, huber_alpha=1e-4,
    xgb_params=None, num_boost_round=400, seed=42,
    w_mid=2.0, w_danger=5.0, w_tail=10.0
):
    base = fit_huber_graph(
        X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test,
        ctx=ctx, tau=tau, alpha=alpha, eps=eps, huber_alpha=huber_alpha
    )

    ytr_all = base["y_train_all"]
    pred_tr_all = base["pred_train_all"]
    mtr = base["mask_train"]
    mte = base["mask_test"]

    res_train = ytr_all - pred_tr_all
    Xgb_train = np.asarray(flatten_window_per_node(X_train_imp), dtype=np.float32)
    Xgb_test  = np.asarray(flatten_window_per_node(X_test_imp),  dtype=np.float32)

    t90 = np.percentile(ytr_all[mtr], 90)
    t95 = np.percentile(ytr_all[mtr], 95)
    t99 = np.percentile(ytr_all[mtr], 99)

    weights = np.ones_like(ytr_all[mtr], dtype=np.float32)
    weights[ytr_all[mtr] >= t90] = w_mid
    weights[ytr_all[mtr] >= t95] = w_danger
    weights[ytr_all[mtr] >= t99] = w_tail

    dtrain = xgb.DMatrix(Xgb_train[mtr], label=res_train[mtr], weight=weights)

    if xgb_params is None:
        xgb_params = {
            "objective": "reg:pseudohubererror",
            "max_depth": 6,
            "eta": 0.05,
            "subsample": 0.80,
            "colsample_bytree": 0.80,
            "lambda": 1.0,
            "tree_method": "hist",
            "seed": seed,
            "verbosity": 0,
        }

    booster = xgb.train(xgb_params, dtrain, num_boost_round=int(num_boost_round))

    res_test_all = booster.predict(xgb.DMatrix(Xgb_test))
    final_test_all = base["pred_test_all"].copy()
    final_test_all[mte] = final_test_all[mte] + res_test_all[mte]

    return {
        "y_true": base["y_test_all"][mte],
        "y_pred_graph": base["pred_test_all"][mte],
        "y_pred_final": final_test_all[mte],
        "gmodel": base["model"],
        "hmodel": booster
    }

DEFAULT_XGB_PARAMS = {
    "objective": "reg:pseudohubererror",
    "max_depth": 6,
    "eta": 0.05,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "lambda": 1.0,
    "tree_method": "hist",
    "verbosity": 0,
}

def _tail_weights(y_obs, w_mid=2.0, w_danger=5.0, w_tail=10.0):
    t90, t95, t99 = (np.percentile(y_obs, p) for p in (90.0, 95.0, 99.0))
    w = np.ones_like(y_obs, dtype=np.float32)
    w[y_obs >= t90] = w_mid
    w[y_obs >= t95] = w_danger
    w[y_obs >= t99] = w_tail
    return w


def _xgb_params(seed, override=None):
    p = dict(DEFAULT_XGB_PARAMS)
    p["seed"] = seed
    if override:
        p.update(override)
    return p


def _residual_stage(ytr_all, pred_tr_all, mtr, pred_te_all, mte,
                    Xgb_train, Xgb_test, xgb_params=None,
                    num_boost_round=400, seed=42,
                    w_mid=2.0, w_danger=5.0, w_tail=10.0):
    res_train = ytr_all - pred_tr_all
    weights = _tail_weights(ytr_all[mtr], w_mid, w_danger, w_tail)
    dtrain = xgb.DMatrix(Xgb_train[mtr], label=res_train[mtr], weight=weights)
    booster = xgb.train(_xgb_params(seed, xgb_params), dtrain,
                        num_boost_round=int(num_boost_round))
    res_test_all = booster.predict(xgb.DMatrix(Xgb_test))
    final = pred_te_all.copy()
    final[mte] = final[mte] + res_test_all[mte]
    return final, booster


def _flat_design(X_train_imp, X_test_imp):
    return (np.asarray(flatten_window_per_node(X_train_imp), dtype=np.float32),
            np.asarray(flatten_window_per_node(X_test_imp), dtype=np.float32))


def fit_flat_hybrid(model, X_train_imp, Y_train, M_train,
                    X_test_imp, Y_test, M_test, **kw):
    Xtr, Xte = _flat_design(X_train_imp, X_test_imp)
    ytr, yte = Y_train.reshape(-1), Y_test.reshape(-1)
    mtr, mte = M_train.reshape(-1) > 0.5, M_test.reshape(-1) > 0.5

    model.fit(Xtr[mtr], ytr[mtr])
    pred_tr_all = np.asarray(model.predict(Xtr), dtype=np.float64)
    pred_te_all = np.asarray(model.predict(Xte), dtype=np.float64)

    final, booster = _residual_stage(ytr, pred_tr_all, mtr, pred_te_all, mte,
                                     Xtr, Xte, **kw)
    return {"y_true": yte[mte], "y_pred_base": pred_te_all[mte],
            "y_pred_final": final[mte], "gmodel": model, "hmodel": booster}


def fit_static_graph_hybrid(model, X_train_imp, Y_train, M_train,
                            X_test_imp, Y_test, M_test,
                            A_static, tau=0, **kw):
    Ztr = make_graph_features_static(X_train_imp, A_static, tau=tau)
    Zte = make_graph_features_static(X_test_imp, A_static, tau=tau)
    Ztr_f = Ztr.reshape(-1, Ztr.shape[-1])
    Zte_f = Zte.reshape(-1, Zte.shape[-1])
    ytr, yte = Y_train.reshape(-1), Y_test.reshape(-1)
    mtr, mte = M_train.reshape(-1) > 0.5, M_test.reshape(-1) > 0.5

    model.fit(Ztr_f[mtr], ytr[mtr])
    pred_tr_all = np.asarray(model.predict(Ztr_f), dtype=np.float64)
    pred_te_all = np.asarray(model.predict(Zte_f), dtype=np.float64)

    Xtr, Xte = _flat_design(X_train_imp, X_test_imp)
    final, booster = _residual_stage(ytr, pred_tr_all, mtr, pred_te_all, mte,
                                     Xtr, Xte, **kw)
    return {"y_true": yte[mte], "y_pred_base": pred_te_all[mte],
            "y_pred_final": final[mte], "gmodel": model, "hmodel": booster}


def fit_tailweighted_xgb(X_train_imp, Y_train, M_train,
                         X_test_imp, Y_test, M_test,
                         xgb_params=None, num_boost_round=400, seed=42,
                         w_mid=2.0, w_danger=5.0, w_tail=10.0,
                         objective="published"):
    Xtr, Xte = _flat_design(X_train_imp, X_test_imp)
    ytr, yte = Y_train.reshape(-1), Y_test.reshape(-1)
    mtr, mte = M_train.reshape(-1) > 0.5, M_test.reshape(-1) > 0.5

    weights = _tail_weights(ytr[mtr], w_mid, w_danger, w_tail)

    if objective == "published":
        model = XGBRegressor(
            n_estimators=200,
            max_depth=8,
            learning_rate=0.07,
            subsample=0.9,
            colsample_bytree=0.8,
            tree_method="hist",
            n_jobs=-1,
            random_state=seed,
        )
        model.fit(Xtr[mtr], ytr[mtr], sample_weight=weights)
        pred = np.asarray(model.predict(Xte), dtype=np.float64)
        booster = model
    elif objective == "pseudohuber":
        prm = _xgb_params(seed, xgb_params)
        prm["base_score"] = float(np.average(ytr[mtr], weights=weights))
        dtrain = xgb.DMatrix(Xtr[mtr], label=ytr[mtr], weight=weights)
        booster = xgb.train(prm, dtrain, num_boost_round=int(num_boost_round))
        pred = np.asarray(booster.predict(xgb.DMatrix(Xte)), dtype=np.float64)
    else:
        raise ValueError(f"unknown objective {objective!r}")

    const = float(np.average(ytr[mtr], weights=weights))
    const_mae = float(np.mean(np.abs(yte[mte] - const)))
    pred_mae = float(np.mean(np.abs(pred[mte] - yte[mte])))
    diverged = (not np.isfinite(pred_mae)) or pred_mae > 2.0 * const_mae
    if diverged:
        print(f"CONTROL Tail-Weighted XGB ({objective}) DIVERGED")
        print(f"MAE {pred_mae:.3f} vs constant-predictor MAE {const_mae:.3f}")
        print(f"pred mean {pred[mte].mean():.3f}  std {pred[mte].std():.6f}")

    return {"y_true": yte[mte], "y_pred": pred[mte], "model": booster,
            "diverged": bool(diverged), "mae": pred_mae,
            "const_mae": const_mae, "objective": objective}

def choose_best_tau_dynamic(train_pack, val_pack, ctx, tau_list):
    rows = []
    best_tau = None
    best_mae = np.inf
    for tau in tau_list:
        if tau >= train_pack["X"].shape[1]:
            continue
        res = fit_huber_graph(
            train_pack["X"], train_pack["Y"], train_pack["M"],
            val_pack["X"], val_pack["Y"], val_pack["M"],
            ctx=ctx, tau=tau
        )
        mae = mean_absolute_error(res["y_true"], res["y_pred"])
        rows.append({"tau": tau, "val_MAE": mae})
        if mae < best_mae:
            best_mae = mae
            best_tau = tau
    return best_tau, pd.DataFrame(rows)

def choose_best_tau_static(train_pack, val_pack, A_static, tau_list):
    rows = []
    best_tau = None
    best_mae = np.inf
    for tau in tau_list:
        if tau >= train_pack["X"].shape[1]:
            continue
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("huber", HuberRegressor(epsilon=1.35, max_iter=500))
        ])
        res = fit_static_graph_regressor(
            model,
            train_pack["X"], train_pack["Y"], train_pack["M"],
            val_pack["X"], val_pack["Y"], val_pack["M"],
            A_static=A_static, tau=tau
        )
        mae = mean_absolute_error(res["y_true"], res["y_pred"])
        rows.append({"tau": tau, "val_MAE": mae})
        if mae < best_mae:
            best_mae = mae
            best_tau = tau
    return best_tau, pd.DataFrame(rows)

def eval_hybrid_params(params, tau, train_pack, val_pack, ctx):
    alpha, eps, log_huber_alpha, max_depth, eta, subsample, colsample, reg_lambda, nround, w_mid, w_danger, w_tail = params
    huber_alpha = 10 ** float(log_huber_alpha)

    hybrid = fit_huber_hybrid(
        train_pack["X"], train_pack["Y"], train_pack["M"],
        val_pack["X"],   val_pack["Y"],   val_pack["M"],
        ctx=ctx,
        tau=tau,
        alpha=float(alpha),
        eps=float(eps),
        huber_alpha=float(huber_alpha),
        xgb_params={
            "objective": "reg:pseudohubererror",
            "max_depth": int(round(max_depth)),
            "eta": float(eta),
            "subsample": float(subsample),
            "colsample_bytree": float(colsample),
            "lambda": float(reg_lambda),
            "tree_method": "hist",
            "seed": RANDOM_SEED,
            "verbosity": 0,
        },
        num_boost_round=int(round(nround)),
        seed=RANDOM_SEED,
        w_mid=float(w_mid),
        w_danger=float(w_danger),
        w_tail=float(w_tail)
    )

    y_true = hybrid["y_true"]
    y_pred = hybrid["y_pred_final"]
    mae = mean_absolute_error(y_true, y_pred)

    thr99 = np.percentile(y_true, 99)
    idx_tail = y_true >= thr99
    if idx_tail.sum() > 0:
        mfb_tail, _ = compute_mfb_nmse(y_true[idx_tail], y_pred[idx_tail])
        return mae + (10.0 * abs(mfb_tail))
    return mae

def sample_population(n, bounds):
    pop = np.random.rand(n, bounds.shape[0])
    return bounds[:, 0] + pop * (bounds[:, 1] - bounds[:, 0])

def clip_params(x, bounds):
    return np.minimum(np.maximum(x, bounds[:, 0]), bounds[:, 1])

JSO_TRACE = []


def eval_hybrid_params_ctrl(params, train_pack, val_pack, variant,
                            A_static=None, tau=0):
    (log_huber_alpha, max_depth, eta, subsample, colsample,
     reg_lambda, nround, w_mid, w_danger, w_tail) = params
    huber_alpha = 10 ** float(log_huber_alpha)

    base = Pipeline([("scaler", StandardScaler()),
                     ("huber", HuberRegressor(epsilon=1.35,
                                              alpha=float(huber_alpha),
                                              max_iter=500))])
    kw = dict(
        xgb_params={
            "objective": "reg:pseudohubererror",
            "max_depth": int(round(max_depth)),
            "eta": float(eta),
            "subsample": float(subsample),
            "colsample_bytree": float(colsample),
            "lambda": float(reg_lambda),
            "tree_method": "hist",
            "seed": RANDOM_SEED,
            "verbosity": 0,
        },
        num_boost_round=int(round(nround)),
        seed=RANDOM_SEED,
        w_mid=float(w_mid), w_danger=float(w_danger), w_tail=float(w_tail),
    )
    args = (train_pack["X"], train_pack["Y"], train_pack["M"],
            val_pack["X"],   val_pack["Y"],   val_pack["M"])
    if variant == "flat":
        r = fit_flat_hybrid(base, *args, **kw)
    elif variant == "static":
        assert A_static is not None
        r = fit_static_graph_hybrid(base, *args, A_static=A_static,
                                    tau=int(tau), **kw)
    else:
        raise ValueError(f"unknown control variant: {variant!r}")

    y_true, y_pred = r["y_true"], r["y_pred_final"]
    if not np.isfinite(y_pred).all():
        return float("inf")
    mae = mean_absolute_error(y_true, y_pred)
    thr99 = np.percentile(y_true, 99)
    idx = y_true >= thr99
    if idx.sum() > 0:
        mfb_tail, _ = compute_mfb_nmse(y_true[idx], y_pred[idx])
        return mae + (10.0 * abs(mfb_tail))
    return mae


def jso_optimize(train_pack, val_pack, ctx, tau, bounds, n_pop=12, iters=15,
                 seed=1, fitness_fn=None, label="dynamic"):
    if fitness_fn is None:
        def fitness_fn(p):
            return eval_hybrid_params(p, tau, train_pack, val_pack, ctx)
    np.random.seed(seed)
    dim = bounds.shape[0]
    pop = sample_population(n_pop, bounds)
    fitness = np.array([fitness_fn(p) for p in pop], dtype=float)
    _n_eval = int(n_pop)

    def _trace(it):
        JSO_TRACE.append(dict(
            label=label, seed=int(seed), dim=int(dim), n_pop=int(n_pop),
            iter=int(it), n_eval=int(_n_eval),
            best=float(np.min(fitness)), worst=float(np.max(fitness)),
            mean=float(np.mean(fitness)), sd=float(np.std(fitness, ddof=0))))
    _trace(0)

    best_idx = np.argmin(fitness)
    best_p = pop[best_idx].copy()
    best_f = float(fitness[best_idx])
    print(f"    initial best objective={best_f:.4f}")

    for it in range(iters):
        c = (1.0 - it / max(iters, 1))
        new_pop = pop.copy()
        for i in range(n_pop):
            if np.random.rand() < 0.5:
                step = np.random.randn(dim) * c * 0.1
                cand = pop[i] + step + c * (best_p - pop[i]) * np.random.rand(dim)
            else:
                j = np.random.randint(0, n_pop)
                step = (pop[j] - pop[i]) * (np.random.rand(dim) - 0.5) * c
                cand = pop[i] + step
            new_pop[i] = clip_params(cand, bounds)

        new_fit = np.array([fitness_fn(p) for p in new_pop], dtype=float)
        _n_eval += int(n_pop)
        improved = new_fit < fitness
        pop[improved] = new_pop[improved]
        fitness[improved] = new_fit[improved]

        best_idx = np.argmin(fitness)
        if fitness[best_idx] < best_f:
            best_f = float(fitness[best_idx])
            best_p = pop[best_idx].copy()

        _trace(it + 1)
        print(f"    iter {it + 1:02d}/{iters:02d}: best objective={best_f:.4f}")

    print(f"    [{label}] dim={dim} evaluations={_n_eval} "
          f"best={best_f:.4f}")
    return best_p, best_f

def add_result_row(rows, tail_rows, H, split_name, model_name, y_true, y_pred, extra=None):
    summary = summarize_regression(y_true, y_pred)
    row = {
        "H": H,
        "split": split_name,
        "model": model_name,
        "MAE": summary["MAE"],
        "RMSE": summary["RMSE"],
        "R2": summary["R2"],
        "n_test": int(len(y_true))
    }
    if extra:
        row.update(extra)
    rows.append(row)

    for p, thr, mae, rmse, n_tail in compute_tail_metrics(y_true, y_pred):
        tail_row = {
            "H": H,
            "split": split_name,
            "model": model_name,
            "percentile": p,
            "threshold": float(thr),
            "tail_MAE": float(mae) if pd.notna(mae) else np.nan,
            "tail_RMSE": float(rmse) if pd.notna(rmse) else np.nan,
            "n_tail": int(n_tail)
        }
        if extra:
            tail_row.update(extra)
        tail_rows.append(tail_row)

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_parquet(DATA_PATH).copy()
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["station_id"] = df["station_id"].astype(str)

stations = (
    df[["station_id", "station_name"]]
    .drop_duplicates()
    .sort_values("station_id")
    .reset_index(drop=True)
)
station_ids = stations["station_id"].tolist()
N = len(station_ids)

candidate_features = ["pm25", "temp_2m", "dewpoint_2m", "surface_pressure", "u10", "v10"]
features = [c for c in candidate_features if c in df.columns]
assert "pm25" in features
pm25_idx = features.index("pm25")
u_idx = features.index("u10")
v_idx = features.index("v10")

all_times = pd.date_range(df["timestamp"].min(), df["timestamp"].max(), freq="h")
base = pd.MultiIndex.from_product([all_times, station_ids], names=["timestamp", "station_id"]).to_frame(index=False)

aligned = base.merge(
    df[["timestamp", "station_id"] + features],
    on=["timestamp", "station_id"],
    how="left"
)

X_feat = []
for feat in features:
    mat = aligned.pivot(index="timestamp", columns="station_id", values=feat).reindex(all_times)[station_ids]
    X_feat.append(mat.to_numpy(dtype=np.float32))

X_all_raw = np.stack(X_feat, axis=-1).astype(np.float32)

pm25_raw = aligned.pivot(index="timestamp", columns="station_id", values="pm25").reindex(all_times)[station_ids]
pm25_raw_np = pm25_raw.to_numpy(dtype=np.float32)
Y_mask_full = (~np.isnan(pm25_raw_np)).astype(np.float32)

train_timeline_mask = all_times < SPLIT_TIME
fill_values = build_fill_values_from_train_timeline(X_all_raw[train_timeline_mask])

nodes, edges_df = build_nodes_edges(df, station_ids, k=GRAPH_K)
A_static = build_static_adj_from_nodes(nodes, k=GRAPH_K, sigma_km=None, self_loops=False, row_normalize=True)

src = edges_df["src"].to_numpy(dtype=int)
dst = edges_df["dst"].to_numpy(dtype=int)
w_dist = edges_df["w_dist"].to_numpy(dtype=np.float32)

src_lat = nodes.loc[src, "lat"].to_numpy()
src_lon = nodes.loc[src, "lon"].to_numpy()
dst_lat = nodes.loc[dst, "lat"].to_numpy()
dst_lon = nodes.loc[dst, "lon"].to_numpy()
edge_bearing = bearing_radians(src_lat, src_lon, dst_lat, dst_lon)

_rng_shuf = np.random.RandomState(12345)
edge_bearing_shuffled = edge_bearing[_rng_shuf.permutation(len(edge_bearing))]

graph_ctx = {
    "src": src,
    "dst": dst,
    "w_dist": w_dist,
    "edge_bearing": edge_bearing,
    "edge_bearing_shuffled": edge_bearing_shuffled,
    "u_idx": u_idx,
    "v_idx": v_idx,
    "N": len(nodes),
}
print(f"graph_ctx ready | GATE_MODE={GATE_MODE!r} | "
      f"{len(src)} edges over {len(nodes)} nodes")

graph_ctx ready | GATE_MODE='fixed' | 32 edges over 8 nodes


In [ ]:
import inspect as _inspect
for _fn_name in ("build_dynamic_adj", "make_graph_features_dynamic"):
    _fn = globals().get(_fn_name)
    assert _fn is not None
    _ps = _inspect.signature(_fn).parameters
    assert "gate_mode" in _ps
print(f"GATE_MODE={GATE_MODE!r}")

all_data = {}

for H in H_LIST:
    Xs, Ys, Ms, ref_times, target_times = [], [], [], [], []

    for t in range(L, len(all_times) - H):
        x_raw = X_all_raw[t - L:t, :, :]
        y = pm25_raw_np[t + H, :]
        m = Y_mask_full[t + H, :]

        if m.sum() == 0:
            continue

        Xs.append(x_raw)
        Ys.append(y)
        Ms.append(m)
        ref_times.append(all_times[t])
        target_times.append(all_times[t + H])

    X = np.stack(Xs).astype(np.float32)
    Y = np.stack(Ys).astype(np.float32)
    M = np.stack(Ms).astype(np.float32)
    ref_times = pd.to_datetime(pd.Series(ref_times))
    target_times = pd.to_datetime(pd.Series(target_times))

    train_idx = (target_times < SPLIT_TIME).to_numpy()
    test_idx  = (target_times >= SPLIT_TIME).to_numpy()

    X_train_raw, Y_train, M_train = X[train_idx], Y[train_idx], M[train_idx]
    X_test_raw,  Y_test,  M_test  = X[test_idx],  Y[test_idx],  M[test_idx]

    X_train_imp = impute_windows(X_train_raw, fill_values)
    X_test_imp  = impute_windows(X_test_raw,  fill_values)

    out_h = os.path.join(OUT_DIR, f"H{H}")
    os.makedirs(out_h, exist_ok=True)
    np.save(os.path.join(out_h, "X_train.npy"), X_train_raw)
    np.save(os.path.join(out_h, "Y_train.npy"), Y_train)
    np.save(os.path.join(out_h, "M_train.npy"), M_train)
    np.save(os.path.join(out_h, "X_test.npy"),  X_test_raw)
    np.save(os.path.join(out_h, "Y_test.npy"),  Y_test)
    np.save(os.path.join(out_h, "M_test.npy"),  M_test)
    np.save(os.path.join(out_h, "X_train_imp.npy"), X_train_imp)
    np.save(os.path.join(out_h, "X_test_imp.npy"),  X_test_imp)

    all_data[H] = {
        "X_train": X_train_raw,
        "Y_train": Y_train,
        "M_train": M_train,
        "X_test": X_test_raw,
        "Y_test": Y_test,
        "M_test": M_test,
        "X_train_imp": X_train_imp,
        "X_test_imp": X_test_imp,
        "ref_times_train": ref_times[train_idx].reset_index(drop=True),
        "ref_times_test": ref_times[test_idx].reset_index(drop=True),
        "target_times_train": target_times[train_idx].reset_index(drop=True),
        "target_times_test": target_times[test_idx].reset_index(drop=True),
    }

    print(f"H={H:>2} | train={X_train_imp.shape[0]:>5} | test={X_test_imp.shape[0]:>5} | "
          f"first train target={all_data[H]['target_times_train'].iloc[0]} | "
          f"last train target={all_data[H]['target_times_train'].iloc[-1]} | "
          f"first test target={all_data[H]['target_times_test'].iloc[0]}")

results_rows = []
tail_rows = []
tau_rows = []
all_models = {}

bounds = np.array([
    [1.0, 8.0],
    [0.01, 0.15],
    [-6.0, -2.0],
    [3.0, 10.0],
    [0.01, 0.20],
    [0.60, 1.00],
    [0.60, 1.00],
    [0.00, 10.00],
    [100.0, 800.0],
    [1.0, 5.0],
    [1.1, 10.0],
    [2.0, 30.0],
], dtype=float)

for H in H_LIST:
    print("\n" + "=" * 80)
    print(f"RUNNING H={H}")
    print("=" * 80)

    pack = all_data[H]
    X_train_imp = pack["X_train_imp"]
    Y_train = pack["Y_train"]
    M_train = pack["M_train"]
    X_test_imp = pack["X_test_imp"]
    Y_test = pack["Y_test"]
    M_test = pack["M_test"]

    train_pack, val_pack = train_val_split_time_order(
        X_train_imp, Y_train, M_train, pack["target_times_train"].to_numpy(), frac=VALID_FRAC
    )

    best_tau_dyn, tau_table_dyn = choose_best_tau_dynamic(train_pack, val_pack, graph_ctx, TAU_LIST)
    best_tau_static, tau_table_static = choose_best_tau_static(train_pack, val_pack, A_static, TAU_LIST)

    tau_table_dyn["H"] = H
    tau_table_dyn["family"] = "dynamic"
    tau_table_static["H"] = H
    tau_table_static["family"] = "static"
    tau_rows.extend(tau_table_dyn.to_dict("records"))
    tau_rows.extend(tau_table_static.to_dict("records"))

    print(f"Selected tau (dynamic) = {best_tau_dyn}")
    print(f"Selected tau (static)  = {best_tau_static}")

    all_models[H] = {
        "selected_tau_dynamic": best_tau_dyn,
        "selected_tau_static": best_tau_static
    }

    ridge = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=10.0))
    ])
    res = fit_flat_regressor(ridge, X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test)
    add_result_row(results_rows, tail_rows, H, "test", "Ridge", res["y_true"], res["y_pred"])
    all_models[H]["ridge"] = res["model"]
    print("Ridge done")

    huber_ng = Pipeline([
        ("scaler", StandardScaler()),
        ("huber", HuberRegressor(epsilon=1.35, max_iter=1000))
    ])
    res = fit_flat_regressor(huber_ng, X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test)
    add_result_row(results_rows, tail_rows, H, "test", "Huber", res["y_true"], res["y_pred"])
    all_models[H]["huber"] = res["model"]
    print("No-Graph Huber done")

    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        n_jobs=-1,
        random_state=RANDOM_SEED
    )
    res = fit_flat_regressor(rf, X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test)
    add_result_row(results_rows, tail_rows, H, "test", "RF", res["y_true"], res["y_pred"])
    all_models[H]["rf"] = res["model"]
    print("RF done")

    xgb_direct = XGBRegressor(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.07,
        subsample=0.9,
        colsample_bytree=0.8,
        tree_method="hist",
        n_jobs=-1,
        random_state=RANDOM_SEED
    )
    res = fit_flat_regressor(xgb_direct, X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test)
    add_result_row(results_rows, tail_rows, H, "test", "XGB", res["y_true"], res["y_pred"])
    all_models[H]["xgb"] = res["model"]
    print("Direct XGB done")

    ridge_static = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=10.0))
    ])
    res = fit_static_graph_regressor(
        ridge_static,
        X_train_imp, Y_train, M_train,
        X_test_imp,  Y_test,  M_test,
        A_static=A_static, tau=best_tau_static
    )
    add_result_row(
        results_rows, tail_rows, H, "test", "Static-Graph Ridge",
        res["y_true"], res["y_pred"], extra={"tau": best_tau_static}
    )
    all_models[H]["static_graph_ridge"] = res["model"]
    print("Static-Graph Ridge done")

    huber_static = Pipeline([
        ("scaler", StandardScaler()),
        ("huber", HuberRegressor(epsilon=1.35, max_iter=1000))
    ])
    res = fit_static_graph_regressor(
        huber_static,
        X_train_imp, Y_train, M_train,
        X_test_imp,  Y_test,  M_test,
        A_static=A_static, tau=best_tau_static
    )
    add_result_row(
        results_rows, tail_rows, H, "test", "Static-Graph Huber",
        res["y_true"], res["y_pred"], extra={"tau": best_tau_static}
    )
    all_models[H]["static_graph_huber"] = res["model"]
    print("Static-Graph Huber done")

    graph_plain = fit_huber_graph(
        X_train_imp, Y_train, M_train,
        X_test_imp,  Y_test,  M_test,
        ctx=graph_ctx, tau=best_tau_dyn, alpha=4.0, eps=0.05, huber_alpha=1e-4
    )
    add_result_row(
        results_rows, tail_rows, H, "test", "Huber Graph",
        graph_plain["y_true"], graph_plain["y_pred"], extra={"tau": best_tau_dyn}
    )
    all_models[H]["graph"] = graph_plain["model"]
    print("Huber Graph done")

    hybrid_plain = fit_huber_hybrid(
        X_train_imp, Y_train, M_train,
        X_test_imp,  Y_test,  M_test,
        ctx=graph_ctx, tau=best_tau_dyn,
        alpha=4.0, eps=0.05, huber_alpha=1e-4,
        num_boost_round=400, seed=RANDOM_SEED,
        w_mid=2.0, w_danger=5.0, w_tail=10.0
    )
    add_result_row(
        results_rows, tail_rows, H, "test", "Hybrid (Graph + XGB residual)",
        hybrid_plain["y_true"], hybrid_plain["y_pred_final"], extra={"tau": best_tau_dyn}
    )
    all_models[H]["hybrid_graph"] = [hybrid_plain["gmodel"], hybrid_plain["hmodel"]]
    print("Hybrid done")

    _ctrl_kw = dict(num_boost_round=400, seed=RANDOM_SEED,
                    w_mid=2.0, w_danger=5.0, w_tail=10.0)

    ctrl_sg_hybrid = fit_static_graph_hybrid(
        Pipeline([("scaler", StandardScaler()),
                  ("huber", HuberRegressor(epsilon=1.35, alpha=1e-4, max_iter=500))]),
        X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test,
        A_static=A_static, tau=best_tau_static, **_ctrl_kw
    )
    add_result_row(
        results_rows, tail_rows, H, "test", "CONTROL Static-Graph Hybrid",
        ctrl_sg_hybrid["y_true"], ctrl_sg_hybrid["y_pred_final"],
        extra={"tau": best_tau_static}
    )
    all_models[H]["ctrl_static_graph_hybrid"] = [ctrl_sg_hybrid["gmodel"],
                                                 ctrl_sg_hybrid["hmodel"]]
    print("CONTROL Static-Graph Hybrid done")

    ctrl_flat_hybrid = fit_flat_hybrid(
        Pipeline([("scaler", StandardScaler()),
                  ("huber", HuberRegressor(epsilon=1.35, alpha=1e-4, max_iter=500))]),
        X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test,
        **_ctrl_kw
    )
    add_result_row(
        results_rows, tail_rows, H, "test", "CONTROL Flat Hybrid (no graph)",
        ctrl_flat_hybrid["y_true"], ctrl_flat_hybrid["y_pred_final"]
    )
    all_models[H]["ctrl_flat_hybrid"] = [ctrl_flat_hybrid["gmodel"],
                                         ctrl_flat_hybrid["hmodel"]]
    print("CONTROL Flat Hybrid done")

    for _tw_obj, _tw_label in (
        ("published", "CONTROL Tail-Weighted XGB"),
        ("pseudohuber", "CONTROL Tail-Weighted XGB (pseudo-Huber)"),
    ):
        ctrl_tw_xgb = fit_tailweighted_xgb(
            X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test,
            objective=_tw_obj, **_ctrl_kw
        )
        _tw_name = _tw_label + (" [DIVERGED]" if ctrl_tw_xgb["diverged"] else "")
        add_result_row(
            results_rows, tail_rows, H, "test", _tw_name,
            ctrl_tw_xgb["y_true"], ctrl_tw_xgb["y_pred"]
        )
        all_models[H]["ctrl_tailweighted_xgb_" + _tw_obj] = ctrl_tw_xgb["model"]
        print(f"{_tw_name} done  (MAE {ctrl_tw_xgb['mae']:.4f}, "
              f"constant predictor {ctrl_tw_xgb['const_mae']:.4f})")

    ctrl_shuf = fit_huber_graph(
        X_train_imp, Y_train, M_train,
        X_test_imp,  Y_test,  M_test,
        ctx=graph_ctx, tau=best_tau_dyn, alpha=4.0, eps=0.05,
        huber_alpha=1e-4, gate_mode="shuffled"
    )
    add_result_row(
        results_rows, tail_rows, H, "test", "CONTROL Shuffled-Bearing Graph",
        ctrl_shuf["y_true"], ctrl_shuf["y_pred"], extra={"tau": best_tau_dyn}
    )
    all_models[H]["ctrl_shuffled_graph"] = ctrl_shuf["model"]
    print("CONTROL Shuffled-Bearing Graph done")

    if RUN_TUNED:
        best_params, best_obj = jso_optimize(
            train_pack=train_pack,
            val_pack=val_pack,
            ctx=graph_ctx,
            tau=best_tau_dyn,
            bounds=bounds,
            n_pop=JSO_POP,
            iters=JSO_ITERS,
            seed=RANDOM_SEED
        )

        alpha, eps, log_huber_alpha, max_depth, eta, subsample, colsample, reg_lambda, nround, w_mid, w_danger, w_tail = best_params
        huber_alpha = 10 ** float(log_huber_alpha)

        tuned_graph = fit_huber_graph(
            X_train_imp, Y_train, M_train,
            X_test_imp,  Y_test,  M_test,
            ctx=graph_ctx, tau=best_tau_dyn,
            alpha=float(alpha), eps=float(eps), huber_alpha=float(huber_alpha)
        )
        add_result_row(
            results_rows, tail_rows, H, "test", "Tuned Huber Graph",
            tuned_graph["y_true"], tuned_graph["y_pred"],
            extra={
                "tau": best_tau_dyn, "alpha": float(alpha), "eps": float(eps),
                "huber_alpha": float(huber_alpha), "val_objective": float(best_obj)
            }
        )
        all_models[H]["tuned_graph"] = tuned_graph["model"]

        tuned_hybrid = fit_huber_hybrid(
            X_train_imp, Y_train, M_train,
            X_test_imp,  Y_test,  M_test,
            ctx=graph_ctx, tau=best_tau_dyn,
            alpha=float(alpha), eps=float(eps), huber_alpha=float(huber_alpha),
            xgb_params={
                "objective": "reg:pseudohubererror",
                "max_depth": int(round(max_depth)),
                "eta": float(eta),
                "subsample": float(subsample),
                "colsample_bytree": float(colsample),
                "lambda": float(reg_lambda),
                "tree_method": "hist",
                "seed": RANDOM_SEED,
                "verbosity": 0,
            },
            num_boost_round=int(round(nround)),
            seed=RANDOM_SEED,
            w_mid=float(w_mid),
            w_danger=float(w_danger),
            w_tail=float(w_tail)
        )
        add_result_row(
            results_rows, tail_rows, H, "test", "Tuned Hybrid (Graph + XGB residual)",
            tuned_hybrid["y_true"], tuned_hybrid["y_pred_final"],
            extra={
                "tau": best_tau_dyn, "alpha": float(alpha), "eps": float(eps),
                "huber_alpha": float(huber_alpha), "max_depth": int(round(max_depth)),
                "eta": float(eta), "subsample": float(subsample),
                "colsample_bytree": float(colsample), "lambda": float(reg_lambda),
                "num_boost_round": int(round(nround)),
                "w_mid": float(w_mid), "w_danger": float(w_danger), "w_tail": float(w_tail),
                "val_objective": float(best_obj)
            }
        )
        all_models[H]["tuned_hybrid_graph"] = [tuned_hybrid["gmodel"], tuned_hybrid["hmodel"]]
        print("Tuned Graph + Tuned Hybrid done")

        bounds_ctrl = bounds[2:, :]
        assert bounds_ctrl.shape[0] == bounds.shape[0] - 2 == 10

        for _cv, _cname, _ctau in (
            ("flat",   "CONTROL Tuned Flat Hybrid (no graph)",   0),
            ("static", "CONTROL Tuned Static-Graph Hybrid",      best_tau_static),
        ):
            print(f"  JSO for {_cname} "
                  f"(dim={bounds_ctrl.shape[0]}, pop={JSO_POP}, "
                  f"iters={JSO_ITERS})")
            _bp, _bo = jso_optimize(
                train_pack=train_pack, val_pack=val_pack,
                ctx=graph_ctx, tau=_ctau, bounds=bounds_ctrl,
                n_pop=JSO_POP, iters=JSO_ITERS, seed=RANDOM_SEED,
                label=_cv,
                fitness_fn=(lambda p, _v=_cv, _t=_ctau:
                            eval_hybrid_params_ctrl(
                                p, train_pack, val_pack, _v,
                                A_static=A_static, tau=_t)),
            )
            (_lha, _md, _eta, _sub, _col, _lam, _nr,
             _wm, _wd, _wt) = _bp
            _ha = 10 ** float(_lha)
            _base = Pipeline([("scaler", StandardScaler()),
                              ("huber", HuberRegressor(epsilon=1.35,
                                                       alpha=float(_ha),
                                                       max_iter=500))])
            _kw = dict(
                xgb_params={"objective": "reg:pseudohubererror",
                            "max_depth": int(round(_md)), "eta": float(_eta),
                            "subsample": float(_sub),
                            "colsample_bytree": float(_col),
                            "lambda": float(_lam), "tree_method": "hist",
                            "seed": RANDOM_SEED, "verbosity": 0},
                num_boost_round=int(round(_nr)), seed=RANDOM_SEED,
                w_mid=float(_wm), w_danger=float(_wd), w_tail=float(_wt))
            _args = (X_train_imp, Y_train, M_train,
                     X_test_imp,  Y_test,  M_test)
            if _cv == "flat":
                _r = fit_flat_hybrid(_base, *_args, **_kw)
            else:
                _r = fit_static_graph_hybrid(_base, *_args,
                                             A_static=A_static,
                                             tau=int(_ctau), **_kw)
            add_result_row(
                results_rows, tail_rows, H, "test", _cname,
                _r["y_true"], _r["y_pred_final"],
                extra={"tau": int(_ctau), "huber_alpha": float(_ha),
                       "max_depth": int(round(_md)), "eta": float(_eta),
                       "subsample": float(_sub),
                       "colsample_bytree": float(_col),
                       "lambda": float(_lam),
                       "num_boost_round": int(round(_nr)),
                       "w_mid": float(_wm), "w_danger": float(_wd),
                       "w_tail": float(_wt),
                       "val_objective": float(_bo),
                       "jso_dim": int(bounds_ctrl.shape[0]),
                       "jso_evals": int(JSO_POP * (JSO_ITERS + 1))})
            all_models[H]["ctrl_tuned_" + _cv + "_hybrid"] = [_r["gmodel"],
                                                              _r["hmodel"]]
            print(f"  {_cname} done (val_obj={_bo:.4f})")

if JSO_TRACE:
    jso_trace_df = pd.DataFrame(JSO_TRACE)
    jso_trace_df.to_csv(os.path.join(OUT_DIR, "jso_convergence.csv"),
                        index=False)
    print(f"\nJSO convergence trace: {len(jso_trace_df)} rows: "
          f"jso_convergence.csv")
    print(jso_trace_df.groupby("label")
          .agg(evals=("n_eval", "max"), best=("best", "min"),
               worst=("worst", "max"), final_mean=("mean", "last"),
               final_sd=("sd", "last")).to_string())
else:
    jso_trace_df = pd.DataFrame()

exceed_rows = []

for H in H_LIST:
    pack = all_data[H]
    X_train_imp = pack["X_train_imp"]
    Y_train = pack["Y_train"]
    M_train = pack["M_train"]
    X_test_imp = pack["X_test_imp"]
    Y_test = pack["Y_test"]
    M_test = pack["M_test"]

    tau = all_models[H]["selected_tau_dynamic"]

    Ztr = make_graph_features_dynamic(X_train_imp, graph_ctx, tau=tau, alpha=4.0, eps=0.05)
    Zte = make_graph_features_dynamic(X_test_imp, graph_ctx, tau=tau, alpha=4.0, eps=0.05)

    Xc_tr = Ztr.reshape(-1, Ztr.shape[-1])
    Xc_te = Zte.reshape(-1, Zte.shape[-1])

    ytr = Y_train.reshape(-1)
    yte = Y_test.reshape(-1)
    mtr = M_train.reshape(-1) > 0.5
    mte = M_test.reshape(-1) > 0.5

    thr = np.percentile(ytr[mtr], 95)
    ytr_cls = (ytr[mtr] >= thr).astype(int)
    yte_cls = (yte[mte] >= thr).astype(int)

    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("logit", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ])
    clf.fit(Xc_tr[mtr], ytr_cls)

    prob = clf.predict_proba(Xc_te[mte])[:, 1]
    pred = (prob >= 0.5).astype(int)

    pr, rc, f1, _ = precision_recall_fscore_support(yte_cls, pred, average="binary", zero_division=0)
    ap = average_precision_score(yte_cls, prob)

    exceed_rows.append({
        "H": H,
        "model": "Graph Exceedance Head (95th pct)",
        "threshold_train_95": float(thr),
        "precision": float(pr),
        "recall": float(rc),
        "f1": float(f1),
        "pr_auc": float(ap),
        "n_test": int(len(yte_cls)),
        "n_positive_test": int(yte_cls.sum())
    })

results_df = pd.DataFrame(results_rows).sort_values(["H", "MAE"]).reset_index(drop=True)
tail_df = pd.DataFrame(tail_rows).sort_values(["H", "model", "percentile"]).reset_index(drop=True)
tau_df = pd.DataFrame(tau_rows).sort_values(["H", "family", "val_MAE"]).reset_index(drop=True)
exceed_df = pd.DataFrame(exceed_rows).sort_values(["H"]).reset_index(drop=True)

results_df.to_csv(os.path.join(OUT_DIR, "all_results_main.csv"), index=False)
tail_df.to_csv(os.path.join(OUT_DIR, "all_results_tail.csv"), index=False)
tau_df.to_csv(os.path.join(OUT_DIR, "selected_tau_validation.csv"), index=False)
exceed_df.to_csv(os.path.join(OUT_DIR, "exceedance_results.csv"), index=False)
joblib.dump(all_models, os.path.join(OUT_DIR, "all_models.pkl"))

print("\nMain results:")
display(results_df)

print("\nTail results (90/95/99 percentiles):")
display(tail_df.head(30))

print("\nValidation tau search:")
display(tau_df)

print("\nExceedance warning results:")
display(exceed_df)

GATE_MODE='fixed'


In [ ]:
from sklearn.metrics import (confusion_matrix, brier_score_loss,
                             precision_recall_curve, roc_auc_score)

exceed_diag_rows = []
reliability_rows = []
sweep_rows = []


def _reliability(y, p, n_bins=10):
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    out = []
    for b in range(n_bins):
        lo, hi = edges[b], edges[b + 1]
        sel = (p >= lo) & (p < hi) if b < n_bins - 1 else (p >= lo) & (p <= hi)
        if sel.sum() == 0:
            continue
        out.append({"bin_lo": float(lo), "bin_hi": float(hi),
                    "n": int(sel.sum()),
                    "mean_pred": float(p[sel].mean()),
                    "observed_freq": float(y[sel].mean())})
    return out


for H in H_LIST:
    pack = all_data[H]
    tau = all_models[H]["selected_tau_dynamic"]

    Ztr = make_graph_features_dynamic(pack["X_train_imp"], graph_ctx, tau=tau,
                                      alpha=4.0, eps=0.05)
    Zte = make_graph_features_dynamic(pack["X_test_imp"], graph_ctx, tau=tau,
                                      alpha=4.0, eps=0.05)
    Xc_tr = Ztr.reshape(-1, Ztr.shape[-1])
    Xc_te = Zte.reshape(-1, Zte.shape[-1])

    ytr, yte = pack["Y_train"].reshape(-1), pack["Y_test"].reshape(-1)
    mtr = pack["M_train"].reshape(-1) > 0.5
    mte = pack["M_test"].reshape(-1) > 0.5

    thr = np.percentile(ytr[mtr], 95)
    ytr_cls = (ytr[mtr] >= thr).astype(int)
    yte_cls = (yte[mte] >= thr).astype(int)

    models = {
        "LogisticRegression": Pipeline([
            ("scaler", StandardScaler()),
            ("logit", LogisticRegression(max_iter=1000, class_weight="balanced"))
        ]),
    }
    pos = max(int(ytr_cls.sum()), 1)
    neg = int(len(ytr_cls) - ytr_cls.sum())
    models["XGBClassifier (scale_pos_weight)"] = XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.8, tree_method="hist",
        scale_pos_weight=float(neg) / float(pos),
        eval_metric="aucpr", n_jobs=-1, random_state=RANDOM_SEED,
    )

    for name, clf in models.items():
        clf.fit(Xc_tr[mtr], ytr_cls)
        prob = np.asarray(clf.predict_proba(Xc_te[mte])[:, 1], dtype=float)
        pred = (prob >= 0.5).astype(int)

        tn, fp, fn, tp = confusion_matrix(yte_cls, pred, labels=[0, 1]).ravel()
        pr, rc, f1, _ = precision_recall_fscore_support(
            yte_cls, pred, average="binary", zero_division=0)

        exceed_diag_rows.append({
            "H": H, "model": name, "threshold_train_95": float(thr),
            "decision_threshold": 0.5,
            "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
            "precision": float(pr), "recall": float(rc), "f1": float(f1),

            "false_alarm_rate": float(fp) / float(fp + tn) if (fp + tn) else np.nan,
            "pr_auc": float(average_precision_score(yte_cls, prob)),
            "roc_auc": float(roc_auc_score(yte_cls, prob))
                        if yte_cls.min() != yte_cls.max() else np.nan,
            "brier": float(brier_score_loss(yte_cls, prob)),
            "base_rate": float(yte_cls.mean()),
            "n_test": int(len(yte_cls)), "n_positive_test": int(yte_cls.sum()),
        })

        for r in _reliability(yte_cls, prob):
            r.update({"H": H, "model": name})
            reliability_rows.append(r)

        for t in np.round(np.arange(0.05, 1.00, 0.05), 2):
            pr_t = (prob >= t).astype(int)
            tn2, fp2, fn2, tp2 = confusion_matrix(
                yte_cls, pr_t, labels=[0, 1]).ravel()
            p2, r2, f2, _ = precision_recall_fscore_support(
                yte_cls, pr_t, average="binary", zero_division=0)
            sweep_rows.append({
                "H": H, "model": name, "threshold": float(t),
                "TP": int(tp2), "FP": int(fp2), "FN": int(fn2), "TN": int(tn2),
                "precision": float(p2), "recall": float(r2), "f1": float(f2),
                "false_alarm_rate": float(fp2) / float(fp2 + tn2) if (fp2 + tn2) else np.nan,
            })

exceed_diag_df = pd.DataFrame(exceed_diag_rows).sort_values(["H", "model"]).reset_index(drop=True)
reliability_df = pd.DataFrame(reliability_rows).reset_index(drop=True)
sweep_df = pd.DataFrame(sweep_rows).reset_index(drop=True)

exceed_diag_df.to_csv(os.path.join(OUT_DIR, "exceedance_diagnostics.csv"), index=False)
reliability_df.to_csv(os.path.join(OUT_DIR, "exceedance_reliability.csv"), index=False)
sweep_df.to_csv(os.path.join(OUT_DIR, "exceedance_threshold_sweep.csv"), index=False)

print("\nExceedance diagnostics (decision threshold 0.5):")
display(exceed_diag_df[["H", "model", "TP", "FP", "FN", "TN", "precision",
                        "recall", "f1", "false_alarm_rate", "pr_auc",
                        "brier", "base_rate"]])

print("\nReliability table:")
display(reliability_df)

print("\nBest F1 per H and model:")
display(sweep_df.loc[sweep_df.groupby(["H", "model"])["f1"].idxmax()]
        .sort_values(["H", "model"]).reset_index(drop=True))